In [1]:
## 1. Load the node initial feature
import os
os.chdir("HyperDC")
import torch
import models
from aggregator import MaxminAggregator
import utils
from data_load import *
pretrain_feature_data = torch.load('data_split/hypergraph_pretrain_noDD.pt')
print(type(pretrain_feature_data))
print(pretrain_feature_data.keys())
node_hypergraph_feat=pretrain_feature_data['node_feat']
print(node_hypergraph_feat)
print(node_hypergraph_feat.shape)

<class 'dict'>
dict_keys(['N_edges', 'N_nodes', 'num_drugs', 'num_diseases', 'NodeEdgePair', 'EdgeNodePair', 'nodewt', 'edgewt', 'node_index_hypergraph', 'edge_index_hypergraph', 'node_feat'])
[[-0.17703886  0.47619486 -0.28882828 ...  0.23660873  0.472763
   0.11794922]
 [ 0.17821127  0.82254696 -0.21597186 ...  0.29461873  0.62663883
   0.58787358]
 [ 0.03682678 -0.1363554  -0.45884526 ...  0.99853772 -0.09837895
   0.93840748]
 ...
 [ 2.64990926  0.34584215 -0.09334952 ...  0.34200341  0.14498441
  -0.60996997]
 [ 1.64587855  1.09300601 -0.27475178 ... -0.37904042 -0.18956095
  -1.27177429]
 [ 2.10951781  0.23628388 -0.17911467 ... -0.7633822  -0.46273226
  -0.10459246]]
(5575, 512)


In [2]:
## 2.Load the optimal model
# Initialization of the model
model = models.multilayers(models.HNHN, [512, 400, 400], \
                1, memory_dim=5575, K=128)
model.load_state_dict(torch.load(f"HyperDC_model/checkpoints/final_model/model_0.pkt", map_location='cuda:0'))
model.to('cuda:0')
model.eval()

# Initialization of the aggregator
cls_layers = [400, 128, 8, 1]
Aggregator = MaxminAggregator(400, cls_layers)
Aggregator.load_state_dict(torch.load(f"HyperDC_model/checkpoints/final_model/Aggregator_0.pkt", map_location='cuda:0'))
Aggregator.to('cuda:0')
Aggregator.eval()

MaxminAggregator(
  (cls): Sequential(
    (0): Linear(in_features=400, out_features=128, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=128, out_features=8, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [3]:
## 3.Load the graph data
import dgl
# Load the full graph data
all_graph= gen_data_simple('hypergraph_pretrain_noDD', gpu=0, dim_edge=400, alpha_e=0, alpha_v=0)
# Load the hyper-edge information (load the training, validation, and test data directly)
data_dict = torch.load(f'data_split/splits/DCsplit0_final.pt')

hyperedge=data_dict['ground_train']+data_dict['ground_valid']
print(len(hyperedge))
# Construct the dgl graph
g = gen_DGLGraph_simple(hyperedge, gpu=0)


26838


In [7]:
# -*- coding: utf-8 -*-
# predict the outcome
import os, csv, ast
import numpy as np
import pandas as pd
import torch
from training import model_eval_simple

# Scale the score
POS_CSV   = "HyperDC_model/predict_outcome/all_positive_score_g.csv"
SCORE_COL = "Score"
LOW_Q, HIGH_Q = 0.01, 0.99

# Save the outcome
OUT1 = "HyperDC_model/predict_outcome/final_outcome/final_MASH_OCA_scaled_index.csv"
OUT2 = "HyperDC_model/predict_outcome/final_outcome/final_MASH_OCA_scaled_name.csv"

NODES_CSV  = "hypergraph_construct/nodes.csv"
ID_DICT_PT = "data_split/hypergraph_pretrain_noDD.pt"


# ★★★★★ Support any number of fixed nodes, for example:
# fixed_indices = [3925]
# fixed_indices = [3925, 1234]
# fixed_indices = [3925, 1234, 999]
fixed_indices = [4674,2267] # hypergraph_idx in hypergraph_construct/node_idx_map.csv

# Automatically avoid all fixed_indices
variable_range = [i for i in range(2774) if i not in fixed_indices]



_EPS = 1e-12

def load_anchors_from_csv(path, score_col=SCORE_COL, low_q=LOW_Q, high_q=HIGH_Q):
    df = pd.read_csv(path)
    if score_col in df.columns:
        col = score_col
    else:
        cands = [c for c in df.columns if c.lower() == score_col.lower()]
        if cands:
            col = cands[0]
        else:
            num_cols = df.select_dtypes(include="number").columns.tolist()
            assert num_cols, f"No numeric column found in {path}"
            col = num_cols[0]
    pos_scores = df[col].astype(float).values
    a = float(np.quantile(pos_scores, low_q))
    b = float(np.quantile(pos_scores, high_q))
    if abs(b - a) < _EPS:
        m = float(np.mean(pos_scores)); a, b = m - _EPS, m + _EPS
    return a, b

def scale_linear(s, a, b):
    return (float(s) - a) / (b - a)

def ensure_dir(p):
    os.makedirs(os.path.dirname(p), exist_ok=True)

def parse_node_tuple(x):
    if isinstance(x, (tuple, list)):
        return tuple(int(v) for v in x)
    return tuple(int(v) for v in ast.literal_eval(str(x)))



a, b = load_anchors_from_csv(POS_CSV)
print(f"[Scaler] anchors from {POS_CSV}: a={a:.9f}, b={b:.9f}")



results = {}

for variable_node in variable_range:

    
    node_indices = fixed_indices + [variable_node]

    score = model_eval_simple(
        all_graph['v_feat'][g.nodes('node')], all_graph['e_feat'][g.nodes('edge')],
        all_graph['v_reg_weight'][g.nodes('node')], all_graph['v_reg_sum'][g.nodes('node')],
        all_graph['e_reg_weight'][g.nodes('edge')],  all_graph['e_reg_sum'][g.nodes('edge')],
        g, model, Aggregator, node_indices, n_layers=1
    )
    s_scaled = scale_linear(score, a, b)
    results[tuple(node_indices)] = (float(score), float(s_scaled))



sorted_results = sorted(results.items(), key=lambda x: x[1][0], reverse=True)
ensure_dir(OUT1)
with open(OUT1, mode='w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Node Indices', 'Score', 'Score_scaled'])
    for nodes, (score, s_scaled) in sorted_results:
        w.writerow([nodes, f"{score:.9f}", f"{s_scaled:.6f}"])
print("-> Wrote:", OUT1)



out_df = pd.read_csv(OUT1)
indices_parsed = out_df['Node Indices'].apply(parse_node_tuple)


max_len = int(indices_parsed.map(len).max())



for i in range(max_len):
    out_df[f'node_idx_{i+1}'] = indices_parsed.apply(
        lambda t, j=i: (t[j] if j < len(t) else np.nan)
    ).astype('Int64')



pt = torch.load(ID_DICT_PT, map_location='cpu')
id_dict = pt['node_index_hypergraph']
inv_map = {int(v): int(k) for k, v in id_dict.items()}

for i in range(max_len):
    col = f'node_idx_{i+1}'
    out_df[f'node{i+1}'] = out_df[col].map(
        lambda x: inv_map.get(int(x)) if pd.notna(x) else pd.NA
    ).astype('Int64')



nodes_df = pd.read_csv(NODES_CSV)
node_index_to_name = dict(zip(nodes_df['node_index'].astype(int), nodes_df['node_name']))

for i in range(max_len):
    out_df[f'Node Name {i+1}'] = out_df[f'node{i+1}'].map(
        lambda x: node_index_to_name.get(int(x)) if pd.notna(x) else pd.NA
    )



final_cols = []
for i in range(max_len):
    final_cols += [f'node{i+1}', f'Node Name {i+1}']
final_cols += ['Score', 'Score_scaled']

result_df = out_df[final_cols]
ensure_dir(OUT2)
result_df.to_csv(OUT2, index=False)
print("-> Wrote:", OUT2)


[Scaler] anchors from /home/hbyu/HyperDC/AHP/predict_outcome/all_positive_score_g.csv: a=0.593961689, b=0.604909391
-> Wrote: /home/hbyu/HyperDC/AHP/predict_outcome/final_outcome/final_MASH_OCA_scaled_index.csv
-> Wrote: /home/hbyu/HyperDC/AHP/predict_outcome/final_outcome/final_MASH_OCA_scaled_name.csv
